# PERTURBATION ANALYSIS - RESULTS

In [ ]:
!pip install -q rouge-score

  Preparing metadata (setup.py) ... done


In [ ]:
# all import here
from google.colab import drive
import os
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, AutoModelForSequenceClassification
from collections import Counter
import torch
import sys
import json
import numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, util
from rouge_score import rouge_scorer


drive.mount('/content/drive')
BASE_PATH = "/content/drive/MyDrive/askqe project official"

Mounted at /content/drive


In [ ]:
PERTURBATION_FIELDS = {
    "alteration": {
        "bt": "ans_bt_noise",
        "direct": "ans_direct_noise"
    },
    "synonym": {
        "bt": "ans_bt_syn",
        "direct": "ans_direct_syn"
    },
    "intensifier": {
        "bt": "ans_bt_int",
        "direct": "ans_direct_int"
    },
    "omission": {
        "bt": "ans_bt_om",
        "direct": "ans_direct_om"
    },
    "word_order": {
        "bt": "ans_bt_wo",
        "direct": "ans_direct_wo"
    },
    "expanction_impact": {
        "bt": "ans_bt_noise",
        "direct": "ans_direct_noise"

    }
}

sbert_model = SentenceTransformer("all-MiniLM-L6-v2")
rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
NLI_MODEL = "cross-encoder/nli-deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
nli_model.to(device)
nli_model.eval()



In [ ]:
def exact_match(ref, pred):
    return 1.0 if str(ref).strip().lower() == str(pred).strip().lower() else 0.0


def f1_score(ref, pred):
    ref_tokens = str(ref).lower().split()
    pred_tokens = str(pred).lower().split()
    if len(ref_tokens) == 0 or len(pred_tokens) == 0:
        return 1.0 if ref_tokens == pred_tokens else 0.0
    common = set(ref_tokens) & set(pred_tokens)
    if len(common) == 0:
        return 0.0
    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(ref_tokens)
    return 2 * (precision * recall) / (precision + recall)


from rouge_score import rouge_scorer
rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def rouge_l(ref, pred):
    return rouge.score(ref, pred)["rougeL"].fmeasure

def sbert_sim(ref, pred):
    emb_ref = sbert_model.encode(ref, convert_to_tensor=True)
    emb_pred = sbert_model.encode(pred, convert_to_tensor=True)
    return util.cos_sim(emb_ref, emb_pred).item()

def nli_label(premise, hypothesis):
    """
    0 = contradiction
    1 = neutral
    2 = entailment
    """
    inputs = tokenizer(
        premise,
        hypothesis,
        return_tensors="pt",
        truncation=True,
        padding=True
    ).to(device)

    with torch.no_grad():
        logits = nli_model(**inputs).logits
        return torch.argmax(logits, dim=1).item()


def is_contradiction(ref, pred):
    return 1.0 if nli_label(ref, pred) == 0 else 0.0


def is_entailment(ref, pred):
    return 1.0 if nli_label(ref, pred) == 2 else 0.0

def evaluate_noise_file(
    baseline_data,
    noise_path,
    field_bt="ans_bt_noise",
    field_direct="ans_direct_noise"
):
    metrics_bt = {
        "em": [],
        "f1": [],
        "rougeL": [],
        "sbert": [],
        "nli_contra": [],
        "nli_entail": []
    }

    metrics_dir = {
        "em": [],
        "f1": [],
        "rougeL": [],
        "sbert": [],
        "nli_contra": [],
        "nli_entail": []
    }

    with open(noise_path, "r", encoding="utf-8") as f:
        for line in tqdm(f, desc=os.path.basename(noise_path)):
            d = json.loads(line)
            ex_id = d["id"]

            if ex_id not in baseline_data:
                continue

            ref = baseline_data.get(ex_id)
            bt = d[field_bt]
            direct = d[field_direct]

            if ref is None or bt is None or direct is None:
                continue


            # ---- BACKTRANSLATION ----
            metrics_bt["em"].append(exact_match(ref, bt))
            metrics_bt["f1"].append(f1_score(ref, bt))
            metrics_bt["rougeL"].append(rouge_l(ref, bt))
            metrics_bt["sbert"].append(sbert_sim(ref, bt))
            metrics_bt["nli_contra"].append(is_contradiction(ref, bt))
            metrics_bt["nli_entail"].append(is_entailment(ref, bt))

            # ---- DIRECT ----
            metrics_dir["em"].append(exact_match(ref, direct))
            metrics_dir["f1"].append(f1_score(ref, direct))
            metrics_dir["rougeL"].append(rouge_l(ref, direct))
            metrics_dir["sbert"].append(sbert_sim(ref, direct))
            metrics_dir["nli_contra"].append(is_contradiction(ref, direct))
            metrics_dir["nli_entail"].append(is_entailment(ref, direct))

    metrics_bt = {k: np.mean(v) for k, v in metrics_bt.items()}
    metrics_dir = {k: np.mean(v) for k, v in metrics_dir.items()}

    return metrics_bt, metrics_dir


def load_baseline(path):
    data = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            d = json.loads(line)
            data[d["id"]] = d["answers"]
    return data


def run_all_perturbations(BASE_PATH):
    baseline_path = os.path.join(BASE_PATH, "QA/mistral-7b/en/Off-en-vanilla.jsonl")
    noise_dir = os.path.join(BASE_PATH, "QA/mistral-7b/es_noise")

    perturbations = {
        "alteration": "alteration_results.jsonl",
        "synonym": "synonym_results.jsonl",
        "word_order": "word_order_results.jsonl",
        "omission": "omission_results.jsonl",
        "intensifier": "intensifier_results.jsonl",
        "expanction_impact": "expanction_impact_results.jsonl",
    }

    baseline = load_baseline(baseline_path)

    results = {}

    for name, file in perturbations.items():
      print(f"\n🧪 Evaluating {name.upper()}")
      path = os.path.join(noise_dir, file)

      fields = PERTURBATION_FIELDS[name]

      bt, direct = evaluate_noise_file(
          baseline_data=baseline,
          noise_path=path,
          field_bt=fields["bt"],
          field_direct=fields["direct"]
      )

      results[name] = {
          "BT": bt,
          "DIRECT": direct
      }

    return results


def print_summary(results):
    print("\n" + "=" * 110)
    print(f"{'PERTURBATION':<15} | {'SBERT':<6} | {'F1':<6} | {'ROUGE':<6} | {'NLI_CONTRA %':<12} | {'NLI_ENTAIL %':<12} | PIPELINE")
    print("=" * 110)

    for name, res in results.items():
        for pipe in ["BT", "DIRECT"]:
            r = res[pipe]
            print(
                f"{name:<15} | "
                f"{r['sbert']:.3f} | "
                f"{r['f1']:.3f} | "
                f"{r['rougeL']:.3f} | "
                f"{r['nli_contra']*100:6.2f}% | "
                f"{r['nli_entail']*100:6.2f}%     | "
                f"{pipe}"
            )
        print("-" * 110)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/738M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
results = run_all_perturbations(BASE_PATH)
print_summary(results)


🧪 Evaluating ALTERATION


alteration_results.jsonl: 970it [02:35,  6.24it/s]



🧪 Evaluating SYNONYM


synonym_results.jsonl: 971it [01:59,  8.15it/s]



🧪 Evaluating WORD_ORDER


...


🧪 Evaluating INTENSIFIER


intensifier_results.jsonl: 971it [01:59,  8.13it/s]



🧪 Evaluating EXPANCTION_IMPACT


expanction_impact_results.jsonl: 971it [01:59,  8.13it/s]


PERTURBATION    | SBERT  | F1     | ROUGE  | NLI_CONTRA % | NLI_ENTAIL % | PIPELINE
alteration      | 0.862 | 0.449 | 0.613 |  48.35% |  21.55%     | BT
alteration      | 0.815 | 0.376 | 0.524 |  35.98% |  27.42%     | DIRECT
--------------------------------------------------------------------------------------------------------------
synonym         | 0.907 | 0.537 | 0.704 |   6.08% |  27.60%     | BT
synonym         | 0.828 | 0.406 | 0.556 |   6.69% |  34.50%     | DIRECT
--------------------------------------------------------------------------------------------------------------
word_order      | 0.917 | 0.540 | 0.707 |   5.19% |  27.49%     | BT
word_order      | 0.834 | 0.411 | 0.560 |   5.91% |  35.37%     | DIRECT
--------------------------------------------------------------------------------------------------------------
omission        | 0.827 | 0.419 | 0.581 |  16.48% |  36.15%     | BT
omission        | 0.764 | 0.329 | 0.469 |  14.73% |  36.77%     | DIRECT
--------------